# Fractional Charges via Golden-Ratio Overlap

CPP v8.0 — 600-cell lattice version

Derives quark fractional charges from time-averaged geometric overlap of inner DP oscillation shell with central qCP, weighted by SSV stress.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from parameters_600cell import phi_inv2, k_curvature, n_events_default

In [ ]:
# Stress profile S(r) ≈ 1/r^4 (inverse-square field squared)
# Amplification γ(r) = 1 + k S(r)
def stress(r, r_source=1.0):
    return 1.0 / (r**4 + 1e-10)  # Avoid singularity

def amplification(r, S):
    return 1 + k_curvature * S

In [ ]:
# Radial overlap integral (spherical volume weighting)
def overlap_delta(r_inner_ratio=phi_inv, n_points=1000):
    r = np.linspace(0.01, 1.0, n_points)           # Normalized cage radius = 1
    dr = r[1] - r[0]
    
    S = stress(r)
    gamma = amplification(r, S)
    weight = gamma * S * r**2                      # dV weighting
    
    r_inner = r_inner_ratio
    inner_mask = r >= r_inner
    
    num = np.sum(weight[inner_mask]) * dr
    den = np.sum(weight) * dr
    
    delta = phi_inv2 * (num / den)
    return delta

In [ ]:
# Single computation
delta = overlap_delta()
print(f"Geometric overlap factor (φ⁻² weighted): {delta:.6f} ≈ 1/3")

In [ ]:
# Monte Carlo with phase/time averaging variation
def mc_overlap(n_events=n_events_default):
    deltas = []
    for _ in range(n_events):
        # Small fluctuation in inner radius (ZBW phase)
        fluct = np.random.normal(1.0, 0.02)  # ±2%
        deltas.append(overlap_delta(r_inner_ratio=phi_inv * fluct))
    return np.mean(deltas), np.std(deltas)

mean_delta, std_delta = mc_overlap()
print(f"\nEnsemble ({n_events_default:,} events):")
print(f"δ = {mean_delta:.6f} ± {std_delta:.6f}")

In [ ]:
# Effective charges
up_charge = +1 - mean_delta
down_charge = -1 + 2 * mean_delta
print(f"\nEffective charges:")
print(f"Up-type: {up_charge:+.4f} e  (target +2/3 ≈ +0.6667)")
print(f"Down-type: {down_charge:+.4f} e  (target -1/3 ≈ -0.3333)")

In [ ]:
# Plot stress-weighted overlap
r = np.linspace(0.01, 1.0, 1000)
S = stress(r)
gamma = amplification(r, S)
weight = gamma * S * r**2

plt.figure(figsize=(8,5))
plt.plot(r, weight, label='Weighted stress $S(r) \\gamma(r) r^2$')
plt.axvline(phi_inv, color='red', linestyle='--', label=f'Inner shell $r = \\phi^{{-1}} \\approx {phi_inv:.3f}$')
plt.fill_between(r[r>=phi_inv], 0, weight[r>=phi_inv], alpha=0.3, color='orange', label='Overlap region')
plt.xlabel('Normalized radius $r / R_{cage}$')
plt.ylabel('Weighted contribution')
plt.title('Golden-Ratio Overlap for Quark Fractional Charge')
plt.legend()
plt.grid(True)
plt.show()